# Football3D — HMR2 on KTH Football II, sequence 2, Camera 1

Attach the private SMPL/HMR2 input and the KTH dataset. Enable Internet and a T4 GPU, then run all cells. The KTH 2D joints define the crop; no tracker JSON is needed.

**HMR2 checkpoint (2.7 GB).** The first run downloads it into `/kaggle/working/hmr2_cache` (resumable: if it stops, rerun cell 3). After that run, **Save Version**, create a **private** Dataset from the `hmr2_cache` output folder, and attach it here — cell 3 then finds it and skips the download entirely.

In [ ]:
from pathlib import Path
import numpy as np

INPUT = Path('/kaggle/input')
position_files = list(INPUT.rglob('positions2d.txt'))
assert position_files, 'Upload positions2d.txt together with the KTH PNG frames'
KTH = position_files[0].parent
camera_dirs = [p for p in INPUT.rglob('*') if p.is_dir() and p.name.lower() == 'camera 1']
# Prefer the actual Camera 1 directory; otherwise locate 00001.png and use its parent.
FRAMES = next((p for p in camera_dirs if len(list(p.glob('*.png'))) >= 175), None)
if FRAMES is None:
    first = next((p for p in INPUT.rglob('*') if p.is_file() and p.name.lower() == '00001.png'), None)
    FRAMES = first.parent if first else None
assert FRAMES is not None, 'KTH PNG frames were not found under /kaggle/input'
frame_files = sorted(FRAMES.glob('*.png')) if FRAMES else []
smpl_files = list(INPUT.rglob('basicmodel_m_lbs_10_207_0_v1.1.0.pkl'))
assert len(frame_files) == 175, f'Expected 175 KTH frames, found {len(frame_files)}'
assert smpl_files, 'Attach the private SMPL dataset containing basicmodel_m_lbs_10_207_0_v1.1.0.pkl'
SMPL_SOURCE = smpl_files[0]
POSITIONS_2D = np.loadtxt(position_files[0]).reshape(-1, 3, 14, 2)[:, 0]
START, END = 1, len(frame_files)
WORK = Path('/kaggle/working/football3d_kth_camera1')
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / 'kth_camera1_pose.npz'
print('KTH:', KTH)
print('Frames:', len(frame_files), 'SMPL:', SMPL_SOURCE)

In [ ]:
import shutil, subprocess, sys

REPO = Path('/kaggle/working/4D-Humans')
if not (REPO / 'hmr2' / '__init__.py').is_file():
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/shubham-goel/4D-Humans.git', str(REPO)], check=True)
assert (REPO / 'hmr2' / '__init__.py').is_file()
# Install the package itself: its setup.py declares every HMR2 inference dependency.
# We use annotated KTH boxes, so the optional Detectron2 detector is not needed.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
# 4D-Humans still imports timm's old module path (same patch as football3d_stage4_ready).
for source in REPO.rglob('*.py'):
    text = source.read_text()
    if 'timm.models.layers' in text:
        source.write_text(text.replace('timm.models.layers', 'timm.layers'))
# The aria2c program (pip's 'aria2' package is not it): parallel, resumable download for cell 3.
if not shutil.which('aria2c'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
sys.path.insert(0, str(REPO))
print('HMR2 checkout:', REPO, '| aria2c:', shutil.which('aria2c'))

In [ ]:
import os, shutil, subprocess, tarfile
from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import DEFAULT_CHECKPOINT

CACHE = Path(CACHE_DIR_4DHUMANS)
checkpoint = Path(DEFAULT_CHECKPOINT)
CKPT_REL = checkpoint.relative_to(CACHE).as_posix()   # logs/train/multiruns/hmr2/0/checkpoints/epoch=35-step=1000000.ckpt

# 1. Fastest: an attached private Dataset that already holds the extracted cache -> no download.
attached = next((p for p in INPUT.rglob(Path(CKPT_REL).name) if p.as_posix().endswith(CKPT_REL)), None)
if attached is not None and not checkpoint.exists():
    source_cache = Path(attached.as_posix()[:-len(CKPT_REL)])
    # Real folders, symlinked files: nothing is copied, and the cache stays writable for SMPL below.
    shutil.copytree(source_cache, CACHE, dirs_exist_ok=True, copy_function=os.symlink)
    print('HMR2 cache linked from attached dataset:', source_cache)

# 2. Otherwise download once, into /kaggle/working so the result can be saved as that Dataset.
# Not hmr2's download_models(): it reads 8 KB at a time over one connection, redraws a progress bar
# after every chunk (~330,000 writes into the notebook), and treats a half-downloaded file as
# finished, so a retry skips the extraction too.
if not checkpoint.exists():
    SAVE = Path('/kaggle/working/hmr2_cache')
    SAVE.mkdir(parents=True, exist_ok=True)
    archive = SAVE / 'hmr2_data.tar.gz'
    subprocess.run(['aria2c', '--continue=true', '-x', '16', '-s', '16', '-k', '1M', '--file-allocation=none',
                    '--show-console-readout=false', '--summary-interval=30', '--console-log-level=warn',
                    '-d', str(SAVE), '-o', archive.name,
                    'https://www.cs.utexas.edu/~pavlakos/4dhumans/hmr2_data.tar.gz'], check=True)
    # aria2 keeps a .aria2 control file until the download is complete; rerunning this cell resumes it.
    assert not Path(str(archive) + '.aria2').exists(), 'Download incomplete: rerun this cell to resume'
    print('Downloaded', round(archive.stat().st_size / 1e9, 2), 'GB. Extracting...')
    with tarfile.open(archive, 'r:*') as bundle:          # 'r:*': the server may send it already decompressed
        bundle.extractall(SAVE, filter='data')
    archive.unlink()                                       # frees 2.7 GB; the extracted files stay
    shutil.copytree(SAVE, CACHE, dirs_exist_ok=True, copy_function=os.symlink)
    print('Extracted to', SAVE, '-> Save Version and make it a private Dataset to skip this next time')

assert checkpoint.exists(), f'HMR2 checkpoint missing: {checkpoint}'

# SMPL last, into ~/.cache only: never into hmr2_cache, which may be saved as a Dataset (SMPL's
# licence forbids redistribution). unlink() first in case the attached cache linked its own copy.
# copytree copies each folder's permissions too, and /kaggle/input folders are read-only.
for folder in [CACHE, *CACHE.rglob('*')]:
    if folder.is_dir() and not folder.is_symlink():
        folder.chmod(0o755)
cache_model = CACHE / 'data' / 'smpl' / 'SMPL_NEUTRAL.pkl'
cache_model.parent.mkdir(parents=True, exist_ok=True)
cache_model.unlink(missing_ok=True)
shutil.copy2(SMPL_SOURCE, cache_model)
print('SMPL copied to:', cache_model)
print('Checkpoint:', checkpoint)

In [ ]:
# Build one tight detector-style box per frame from Camera 1's annotated 2D joints.
boxes = {}
for frame in range(START, END + 1):
    xy = POSITIONS_2D[frame - 1]
    x1, y1 = xy.min(axis=0); x2, y2 = xy.max(axis=0)
    w, h = x2 - x1, y2 - y1
    boxes[frame] = np.array([x1 - .10*w, y1 - .10*h, x2 + .10*w, y2 + .10*h], dtype=np.float32)
print('Boxes:', len(boxes), 'example:', boxes[1])

In [ ]:
import cv2, torch
from hmr2.models import load_hmr2
from hmr2.datasets.vitdet_dataset import ViTDetDataset, DEFAULT_MEAN, DEFAULT_STD
from hmr2.utils import recursive_to
from hmr2.utils.renderer import Renderer

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle settings'
device = torch.device('cuda')
original_torch_load = torch.load
def trusted_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_torch_load(*args, **kwargs)
torch.load = trusted_torch_load
try: model, model_cfg = load_hmr2(str(checkpoint))
finally: torch.load = original_torch_load
model = model.to(device).eval()
renderer = Renderer(model_cfg, faces=model.smpl.faces)
preview_path = WORK / 'kth_camera1_preview.mp4'
preview = subprocess.Popen(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', '512x256', '-r', '25', '-i', '-', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(preview_path)], stdin=subprocess.PIPE)
frames_out, joints_out, vertices_out = [], [], []
orientations_out, body_poses_out, betas_out = [], [], []
for index, frame in enumerate(sorted(boxes), 1):
    image = cv2.imread(str(FRAMES / f'{frame:05d}.png'))
    dataset = ViTDetDataset(model_cfg, image, boxes[frame][None])
    batch = recursive_to(next(iter(torch.utils.data.DataLoader(dataset, batch_size=1))), device)
    with torch.inference_mode():
        result = model(batch); params = result['pred_smpl_params']
        male = model.smpl(global_orient=params['global_orient'].float(), body_pose=params['body_pose'].float(), betas=torch.zeros_like(params['betas']).float(), pose2rot=False)
    vertices = male.vertices[0]
    joints = torch.einsum('jk,kv->jv', model.smpl.J_regressor.to(device), vertices)
    input_patch = batch['img'][0].cpu() * (DEFAULT_STD[:, None, None] / 255) + (DEFAULT_MEAN[:, None, None] / 255)
    rendered = renderer(vertices.cpu().numpy(), result['pred_cam_t'][0].cpu().numpy(), batch['img'][0])
    preview.stdin.write(np.clip(255 * np.concatenate([input_patch.permute(1, 2, 0).numpy(), rendered], axis=1), 0, 255).astype(np.uint8).tobytes())
    frames_out.append(frame); joints_out.append(joints.cpu().numpy()); vertices_out.append(vertices.cpu().numpy())
    orientations_out.append(params['global_orient'][0].cpu().numpy()); body_poses_out.append(params['body_pose'][0].cpu().numpy()); betas_out.append(params['betas'][0].cpu().numpy())
    if index == 1 or index % 10 == 0: print(f'Processed {index}/{len(boxes)}')
preview.stdin.close(); assert preview.wait() == 0
np.savez_compressed(OUT, frame=np.asarray(frames_out), joints=np.asarray(joints_out), vertices=np.asarray(vertices_out), global_orient_rotmat=np.asarray(orientations_out), body_pose_rotmat=np.asarray(body_poses_out), betas=np.asarray(betas_out), fps=np.float32(25))
print('Saved:', OUT, 'Preview:', preview_path)

In [ ]:
from IPython.display import FileLink, Video, display
display(Video(str(preview_path), embed=True))
display(FileLink(str(OUT)))
display(FileLink(str(preview_path)))